# CliniqAI — Router Stage 2 (modality classifier) v2 — full retrain, BreakHis breast_histology source

**Why this exists:** live testing after the gatekeeper v4 fix surfaced a real-world routing bug
— a crisp breast_histology image was classified as `skin_lesion` at 99.8% confidence. This is
the same root-cause pattern as the gatekeeper bug: Stage 2's `breast_histology` training data was
never updated, so it's still the old 50x50px IDC patches. A model trained on tiny blocky patches
has no real basis for recognizing full-resolution H&E histology tissue, and apparently defaults
to whichever modality's color/texture profile is closest in its training distribution.

**This also directly touches the other known Stage 2 bug** (kidney_ct/brain_mri confusion,
documented from an earlier session) since both live in the same 7-way classifier and both need
Stage 2 retrained regardless. This notebook does **not** add deliberate hard-negative-pair
training between kidney_ct/brain_mri — that's a more involved fix. What it does do is retrain
on the same corrected per-modality data used for the gatekeeper v4 fix, and print a full
confusion matrix so you can see plainly whether kidney/brain_mri improved, stayed the same, or
got worse, rather than assuming.

**Data sources:** identical mapping to `gatekeeper_v4_retrain.ipynb` — same 7 specialist
folders, same 12,000-per-modality cap, `breast_histology` pointed at the corrected, deduped
BreakHis data instead of the old IDC patches. No negative/non-medical class here — Stage 2 only
runs on images the gatekeeper already accepted as medical.

Run top to bottom.

## Step 0 — Imports

In [ ]:
import os
import sys
import json
import time
import shutil
import random
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())


## Step 1 — Config

In [ ]:
PROJECT_ROOT = Path(r"E:\programming\CliniqAI")
DATA_RAW = PROJECT_ROOT / "data" / "raw"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
MODELS_DIR = PROJECT_ROOT / "models" / "router"
BACKEND_MODELS_DIR = PROJECT_ROOT / "backend" / "models" / "router"
SANITY_DIR = DATA_RAW / "manual_sanity_check"

# same mapping as gatekeeper_v4_retrain.ipynb -- keep the class order matching the existing
# deployed checkpoint's class_names so nothing downstream needs to change
MODALITY_SOURCES = {
    "bone_fracture": DATA_RAW / "bone_fracture",
    "brain_mri": DATA_RAW / "brain_tumor",
    "breast_histology": DATA_RAW / "breast_histology_v2",   # <-- the fix
    "kidney_ct": DATA_RAW / "kidney_stone",
    "pneumonia": DATA_RAW / "pneumonia",
    "retinal": DATA_RAW / "diabetic_retinopathy",
    "skin_lesion": DATA_RAW / "skin_lesion",
}
CAP_PER_MODALITY = 12000

IMG_SIZE = 224
BATCH_SIZE = 64
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 4
EARLY_STOP_PATIENCE = 8
MAX_EPOCHS = 30
SEED = 42
CHECKPOINT_NAME = "router_stage2_v2_efficientnet_b0"

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMG_EXTS = ("*.png", "*.PNG", "*.jpg", "*.JPG", "*.jpeg", "*.JPEG")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True

MODELS_DIR.mkdir(parents=True, exist_ok=True)
BACKEND_MODELS_DIR.mkdir(parents=True, exist_ok=True)
print("device:", device)


## Step 2 — Glob helper (dedup-safe, same Windows case-insensitivity fix as before)

In [ ]:
def glob_images(folder, exts=IMG_EXTS):
    seen = set()
    out = []
    for ext in exts:
        for p in folder.glob(f"**/{ext}"):
            rp = p.resolve()
            if rp not in seen:
                seen.add(rp)
                out.append(p)
    return out


## Step 3 — Index all 7 modality sources (capped 12,000 each)

In [ ]:
records = []  # (path, modality_tag)
rng = random.Random(SEED)

for modality, folder in MODALITY_SOURCES.items():
    if not folder.exists():
        print(f"WARNING: {folder} not found, skipping {modality}")
        continue
    imgs = glob_images(folder)
    if len(imgs) > CAP_PER_MODALITY:
        imgs = rng.sample(imgs, CAP_PER_MODALITY)
    for p in imgs:
        records.append((str(p), modality))
    print(f"{modality}: {len(imgs)} images (source: {folder.name})")

print(f"\nTotal indexed: {len(records)}")


## Step 4 — Corrupted-file validation pass

In [ ]:
bad_paths = set()
for path, _ in records:
    try:
        with Image.open(path) as im:
            im.load()
    except Exception:
        bad_paths.add(path)

if bad_paths:
    print(f"Dropping {len(bad_paths)} corrupted/unreadable files.")
    records = [r for r in records if r[0] not in bad_paths]
else:
    print("No corrupted files found.")
print(f"{len(records)} usable images remain.")


## Step 5 — Per-modality train/val/test split (80/10/10)

In [ ]:
by_modality = defaultdict(list)
for r in records:
    by_modality[r[1]].append(r)

train_records, val_records, test_records = [], [], []
for modality, recs in by_modality.items():
    recs = recs[:]
    random.Random(SEED).shuffle(recs)
    n = len(recs)
    n_train = int(n * 0.8)
    n_val = int(n * 0.1)
    train_records.extend(recs[:n_train])
    val_records.extend(recs[n_train:n_train + n_val])
    test_records.extend(recs[n_train + n_val:])

random.Random(SEED).shuffle(train_records)
random.Random(SEED).shuffle(val_records)
random.Random(SEED).shuffle(test_records)

print(f"train: {len(train_records)} | val: {len(val_records)} | test: {len(test_records)}")
for modality in sorted(by_modality.keys()):
    tr = sum(1 for r in train_records if r[1] == modality)
    va = sum(1 for r in val_records if r[1] == modality)
    te = sum(1 for r in test_records if r[1] == modality)
    print(f"  {modality}: {tr} / {va} / {te}")


## Step 6 — Write the standalone Dataset module

In [ ]:
dataset_module_path = SCRIPTS_DIR / "router_stage2_dataset_v2.py"
dataset_module_code = '''"""
Standalone, importable Dataset for Router Stage 2 (modality classifier) v2 training.
Must live in its own module (not a notebook cell) for Windows DataLoader multiprocessing
(num_workers > 0) to work -- worker subprocesses re-import this module fresh, so
ImageFile.LOAD_TRUNCATED_IMAGES must be set here at module level, not just in the notebook.
"""
from pathlib import Path
from PIL import Image, ImageFile
from torch.utils.data import Dataset

ImageFile.LOAD_TRUNCATED_IMAGES = True


class RouterStage2DatasetV2(Dataset):
    """records: list of (path, modality_str)."""

    def __init__(self, records, label_to_idx, transform=None):
        self.records = records
        self.label_to_idx = label_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def _load(self, idx):
        path, modality_str = self.records[idx]
        img = Image.open(path).convert("RGB")
        img.load()
        label = self.label_to_idx[modality_str]
        return img, label

    def __getitem__(self, idx):
        try:
            img, label = self._load(idx)
        except (OSError, Image.DecompressionBombError):
            fallback_idx = (idx + 1) % len(self.records)
            img, label = self._load(fallback_idx)
        if self.transform:
            img = self.transform(img)
        return img, label
'''

dataset_module_path.write_text(dataset_module_code, encoding="utf-8")
print(f"Wrote {dataset_module_path}")

sys.path.insert(0, str(SCRIPTS_DIR))
from router_stage2_dataset_v2 import RouterStage2DatasetV2


## Step 7 — Labels, transforms

In [ ]:
class_names = ["bone_fracture", "brain_mri", "breast_histology", "kidney_ct", "pneumonia", "retinal", "skin_lesion"]
label_to_idx = {c: i for i, c in enumerate(class_names)}
print("class_names:", class_names)

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_label_counts = Counter(r[1] for r in train_records)
class_counts = np.array([train_label_counts.get(c, 0) for c in class_names], dtype=np.float64)
class_weights = (class_counts.sum() / (len(class_names) * class_counts)).astype(np.float32)
print("train class counts:", dict(zip(class_names, class_counts.astype(int))))
print("class weights (for loss):", dict(zip(class_names, class_weights)))


## Step 8 — Datasets & DataLoaders (weighted sampler for the modality imbalance)

In [ ]:
train_ds = RouterStage2DatasetV2(train_records, label_to_idx, transform=train_transform)
val_ds = RouterStage2DatasetV2(val_records, label_to_idx, transform=eval_transform)
test_ds = RouterStage2DatasetV2(test_records, label_to_idx, transform=eval_transform)

sample_weights = np.array([class_weights[label_to_idx[r[1]]] for r in train_records], dtype=np.float32)
sampler = torch.utils.data.WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True, prefetch_factor=4,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True, prefetch_factor=4,
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True, prefetch_factor=4,
)
print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")


## Step 9 — Model (wide head, matches the existing Stage 2 checkpoint format)

In [ ]:
def build_model(num_classes, hidden_dim=512):
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, hidden_dim),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(hidden_dim, num_classes),
    )
    return model

model = build_model(len(class_names)).to(device, memory_format=torch.channels_last)
print(model.classifier)


## Step 10 — Loss, optimizer, scheduler, AMP

In [ ]:
criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, device=device))
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler(device=device.type, enabled=use_amp)


## Step 11 — Training loop

Per-epoch checkpointing on val_loss improvement, resume cell below for kernel-restart recovery.

In [ ]:
checkpoint_inprogress_path = MODELS_DIR / f"{CHECKPOINT_NAME}.checkpoint_inprogress.pt"
history = []
best_val_loss = float("inf")
epochs_no_improve = 0
start_epoch = 0

def run_epoch(loader, train_mode):
    model.train(train_mode)
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(device, memory_format=torch.channels_last, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if train_mode:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train_mode):
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                outputs = model(images)
                loss = criterion(outputs, labels)

            if train_mode:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)

    return total_loss / total, correct / total

for epoch in range(start_epoch, MAX_EPOCHS):
    t0 = time.time()
    train_loss, train_acc = run_epoch(train_loader, train_mode=True)
    val_loss, val_acc = run_epoch(val_loader, train_mode=False)
    scheduler.step(val_loss)
    elapsed = time.time() - t0

    print(f"epoch {epoch+1:02d} | train_loss {train_loss:.4f} acc {train_acc:.4f} "
          f"| val_loss {val_loss:.4f} acc {val_acc:.4f} | {elapsed:.1f}s")

    history.append({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc,
                     "val_loss": val_loss, "val_acc": val_acc})

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "class_names": class_names,
            "classes": class_names,
            "num_classes": len(class_names),
            "architecture": "efficientnet_b0",
            "epoch": epoch + 1,
            "best_val_loss": best_val_loss,
        }, checkpoint_inprogress_path)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"Early stopping at epoch {epoch+1} (no improvement for {EARLY_STOP_PATIENCE} epochs).")
            break

print("Training done. Best val_loss:", best_val_loss)


## Step 12 — Resume from checkpoint (only if the kernel was restarted mid-training)

In [ ]:
# resume_ckpt = torch.load(checkpoint_inprogress_path, map_location=device)
# model.load_state_dict(resume_ckpt["model_state_dict"])
# start_epoch = resume_ckpt["epoch"]
# best_val_loss = resume_ckpt["best_val_loss"]
# print(f"Resumed from epoch {start_epoch}, best_val_loss={best_val_loss:.4f}")
# # then re-run the Step 11 training loop cell, which will pick up from start_epoch


## Step 13 — Final save (project root + backend copy)

In [ ]:
best_ckpt = torch.load(checkpoint_inprogress_path, map_location=device)
model.load_state_dict(best_ckpt["model_state_dict"])

final_path = MODELS_DIR / f"{CHECKPOINT_NAME}.pt"
torch.save(best_ckpt, final_path)

backend_final_path = BACKEND_MODELS_DIR / f"{CHECKPOINT_NAME}.pt"
shutil.copy2(final_path, backend_final_path)

history_path = MODELS_DIR / f"{CHECKPOINT_NAME}.history.json"
history_path.write_text(json.dumps(history, indent=2), encoding="utf-8")

print("Saved:", final_path)
print("Saved:", backend_final_path)
print("Saved:", history_path)


## Step 14 — Per-modality test accuracy + full confusion matrix

The confusion matrix is the actual point of this eval — it directly shows whether
breast_histology stopped being misrouted to skin_lesion, and separately, whether kidney_ct/
brain_mri confusion (the other known Stage 2 bug) improved, stayed the same, or got worse. Don't
just look at overall accuracy.

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, memory_format=torch.channels_last)
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            outputs = model(images)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
overall_acc = (all_preds == all_labels).mean()
print(f"Overall test accuracy: {overall_acc:.4f} (n={len(all_labels)})")

print("\nPer-modality accuracy:")
for i, c in enumerate(class_names):
    idx = np.where(all_labels == i)[0]
    if len(idx) == 0:
        continue
    acc = (all_preds[idx] == all_labels[idx]).mean()
    print(f"  {c}: {acc:.4f} (n={len(idx)})")

n = len(class_names)
cm = np.zeros((n, n), dtype=int)
for t, p in zip(all_labels, all_preds):
    cm[t, p] += 1

print("\nConfusion matrix (rows=true, cols=predicted):")
header = "true\\pred".ljust(18) + "".join(c[:10].ljust(11) for c in class_names)
print(header)
for i, c in enumerate(class_names):
    row = c.ljust(18) + "".join(str(cm[i, j]).ljust(11) for j in range(n))
    print(row)

print("\nSpecific pairs to check:")
bh_idx, sl_idx = label_to_idx["breast_histology"], label_to_idx["skin_lesion"]
kc_idx, bm_idx = label_to_idx["kidney_ct"], label_to_idx["brain_mri"]
print(f"  breast_histology mistaken for skin_lesion: {cm[bh_idx, sl_idx]} / {cm[bh_idx].sum()}")
print(f"  kidney_ct mistaken for brain_mri: {cm[kc_idx, bm_idx]} / {cm[kc_idx].sum()}")
print(f"  brain_mri mistaken for kidney_ct: {cm[bm_idx, kc_idx]} / {cm[bm_idx].sum()}")


## Step 15 — Real-world sanity check

Runs whatever's in `manual_sanity_check/` through the full Stage 2 model directly — this is the
test that actually matters, same discipline as the gatekeeper fix. The breast_histology
Wikimedia image should now predict `breast_histology`, not `skin_lesion`.

In [ ]:
sanity_images = glob_images(SANITY_DIR)
print(f"Found {len(sanity_images)} sanity images in {SANITY_DIR}")

model.eval()
for img_path in sanity_images:
    try:
        img = Image.open(img_path).convert("RGB")
    except Exception as e:
        print(f"{img_path.name}: failed to open ({e})")
        continue
    x = eval_transform(img).unsqueeze(0).to(device, memory_format=torch.channels_last)
    with torch.no_grad():
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            probs = torch.softmax(model(x), dim=1)[0]
    pred_idx = probs.argmax().item()
    print(f"\n{img_path.name}: predicted {class_names[pred_idx]} ({probs[pred_idx].item():.4f})")
    top3 = probs.argsort(descending=True)[:3]
    for i in top3:
        print(f"    {class_names[i]}: {probs[i].item():.4f}")


## Notes

- **Do not point `backend/app/config.py`'s `ROUTER_STAGE2_PATH` at v2 until Step 15 confirms**
  the breast_histology sanity image routes correctly — same discipline as every other checkpoint
  in this project.
- Check the confusion matrix's kidney_ct/brain_mri numbers specifically (Step 14's "specific
  pairs" printout). If that confusion is still present after this retrain, it means better
  per-modality data alone isn't enough for that pair and it genuinely needs deliberate
  hard-negative-pair training (weighted sampling of confusable pairs, or a contrastive loss term)
  — a separate, more involved notebook, not a quick follow-up to this one.
- Worth committing `scripts/router_stage2_dataset_v2.py` and this notebook to `feature/router`
  once trained, regardless of how the kidney/brain_mri numbers land.
- `models/router/router_stage2_v2_efficientnet_b0.pt` (+ backend copy) needs uploading to the
  shared Google Drive once validated.